# Pandas 缺失值清洗独立练习：电商订单数据

## 1. 业务背景

某电商平台从多个业务系统中汇总了一批订单数据。

由于数据来自不同渠道，字段中存在：

- `None` 和 `pd.NA`
- 空字符串和纯空格
- 前后空格
- 大小写不统一
- `N/A`
- `NULL`
- `none`
- `unknown`
- `NaN`
- `-`
- 无效日期
- 应为数值但实际保存为字符串的数据

本练习要求独立完成从原始数据检查、清洗规则制定，到缺失报告和最终验证的完整流程。

---

## 2. 数据字段

| 字段 | 业务含义 | 预期类型 |
|---|---|---|
| `order_id` | 订单编号 | 字符串 |
| `customer_id` | 客户编号 | 字符串 |
| `region` | 订单所属区域 | 字符串 |
| `order_status` | 订单状态 | 字符串 |
| `payment_amount` | 订单支付金额 | 浮点数 |
| `sales_rep` | 负责销售人员 | 字符串 |
| `promo_code` | 优惠码 | 字符串 |
| `order_time` | 下单时间 | 日期时间 |

---

## 3. 业务规则

### 3.1 `order_id`

- 清除首尾空格。
- 统一转换为大写。
- 合法格式为 `ORD-` 加四位数字，例如 `ORD-1001`。
- 缺失订单编号的记录无法追踪，应保存问题记录后删除整行。

### 3.2 `customer_id`

- 清除首尾空格。
- 统一转换为大写。
- 正常客户编号格式为 `C` 加三位数字，例如 `C001`。
- 客户编号缺失表示游客下单，不删除记录，最终填充为 `GUEST`。

### 3.3 `region`

- 清除首尾空格。
- 统一转换为大写。
- 合法区域包括：

```text
NORTH
SOUTH
EAST
WEST
CENTRAL
```

- 区域缺失时保留订单，最终填充为 `UNKNOWN`。

### 3.4 `order_status`

- 清除首尾空格。
- 统一转换为大写。
- 合法状态包括：

```text
PAID
PENDING
CANCELLED
REFUNDED
```

- 状态缺失时最终填充为 `UNKNOWN`。

### 3.5 `payment_amount`

- 清除首尾空格。
- 将伪缺失值统一为真正缺失值。
- 转换为数值类型。
- 转换失败的非标准值统一转为缺失值。
- 缺失金额暂时保留，不删除、不填充。
- 非缺失金额不得小于 `0`。

### 3.6 `sales_rep`

- 清除首尾空格。
- 统一转换为大写。
- 缺失销售人员最终填充为 `UNASSIGNED`。

### 3.7 `promo_code`

- 清除首尾空格。
- 统一转换为大写。
- 缺失、`none`、`unknown` 等均表示未使用优惠码。
- 最终填充为 `NO_PROMO`。

### 3.8 `order_time`

- 转换为日期时间类型。
- 无效日期和无法解析的文本统一转换为 `NaT`。
- 下单时间是订单分析的关键字段。
- 时间缺失或转换失败的记录，应保存问题记录后删除整行。

---

## 4. 练习任务

1. 保留原始 DataFrame，创建独立清洗副本。
2. 检查数据规模、字段类型和 Pandas 当前识别出的缺失值。
3. 使用适当方法检查重点字段的原始表达。
4. 根据检查结果自行定义伪缺失值规则。
5. 完成所有文本字段的格式标准化。
6. 将 `payment_amount` 转换为数值类型。
7. 将 `order_time` 转换为日期时间类型。
8. 保存并删除关键字段缺失的记录。
9. 删除记录后、业务填充前，生成全表缺失报告。
10. 根据业务规则处理其余字段的缺失值。
11. 验证类别字段只包含合法值。
12. 验证编号字段符合规定格式。
13. 验证金额字段为数值类型且不存在负数。
14. 验证时间字段转换成功。
15. 验证保留记录与删除记录能够完整对应原始数据。

---

## 5. 缺失报告要求

缺失报告命名为：

```text
missing_report
```

至少包含以下三列：

| 字段 | 含义 |
|---|---|
| `column` | 原字段名 |
| `missing_count` | 缺失数量 |
| `missing_rate` | 缺失比例 |

要求：

- 缺失比例保留四位小数；
- 按缺失数量降序排列；
- 缺失数量相同时按字段名升序排列；
- 报告必须在业务填充之前生成。

---

## 6. 限制条件

- 不允许直接修改 `df_raw`。
- 不允许手工逐行修改数据。
- 不允许直接对整个 DataFrame 使用无条件的 `dropna()`。
- 不允许用一个统一值填充所有字段。
- 删除记录之前必须保存被删除记录。
- 合法范围必须来自题目给出的业务规则，不能根据当前数据的 `unique()` 结果自行生成。
- 最终必须使用 `assert` 或等价布尔条件进行自动验证。

In [2]:
import pandas as pd

data = {
    "order_id": [
        "ord-1001",
        "ORD-1002",
        " ORD-1003 ",
        "ORD-1004",
        " ",
        "ord-1006",
        "ORD-1007",
        "ORD-1008",
        "ORD-1009",
        None,
        "ORD-1011",
        "ORD-1012",
        "ord-1013",
        "ORD-1014",
        " ORD-1015 ",
        "ORD-1016",
        "ORD-1017",
        "ORD-1018",
        "ORD-1019",
        "ORD-1020"
    ],

    "customer_id": [
        "C001",
        " C002 ",
        None,
        "N/A",
        "C005",
        "C006",
        "null",
        "C008",
        "C009",
        None,
        "C011",
        "",
        "C013",
        "C014",
        "C015",
        " none ",
        "C017",
        "C018",
        pd.NA,
        "C020"
    ],

    "region": [
        "North",
        " north ",
        "SOUTH",
        "south ",
        "East",
        " east ",
        None,
        "West",
        " west ",
        "central",
        " Central ",
        "N/A",
        "NORTH",
        "-",
        "SOUTH",
        "EAST",
        "WEST",
        "CENTRAL",
        "north",
        "SOUTH"
    ],

    "order_status": [
        "paid",
        " PAID ",
        "pending",
        None,
        "CANCELLED",
        " refunded ",
        "-",
        "PAID",
        "pending",
        "unknown",
        "PAID",
        "NULL",
        "REFUNDED",
        "cancelled",
        " PAID ",
        "pending",
        "refunded",
        " ",
        "PAID",
        "CANCELLED"
    ],

    "payment_amount": [
        "1299.50",
        "850.00",
        "NULL",
        "399.90",
        "-",
        "199.00",
        "0",
        None,
        "760.50",
        "500",
        "1250",
        "NaN",
        "320.75",
        "999.99",
        " 450.00 ",
        "NULL",
        "210.00",
        "89.90",
        "-",
        "600.00"
    ],

    "sales_rep": [
        "Li",
        "li",
        "Wang",
        " wang ",
        "Chen",
        "NULL",
        "Zhao",
        " zhao ",
        "N/A",
        "Sun",
        "sun",
        "",
        "Li",
        None,
        "Wang",
        "Chen",
        "Zhao",
        "Sun",
        "-",
        "Li"
    ],

    "promo_code": [
        "SUMMER10",
        " ",
        "none",
        "VIP20",
        "",
        "N/A",
        "WELCOME",
        None,
        "-",
        "NONE",
        "SUMMER10",
        "NULL",
        "   ",
        "VIP20",
        "unknown",
        "WELCOME",
        "NONE",
        "N/A",
        "FLASH5",
        None
    ],

    "order_time": [
        "2026-07-02 09:00",
        "2026-07-02 09:15",
        "2026-07-02 09:30",
        "2026-07-02 09:45",
        "2026-07-02 10:00",
        "2026-07-02 10:15",
        "2026-07-02 10:30",
        "2026-02-30 14:20",
        "2026-07-02 11:00",
        "2026-07-02 11:15",
        "2026-07-02 11:30",
        "2026-07-02 11:45",
        "2026-07-02 12:00",
        "2026-07-02 12:15",
        "2026-07-02 12:30",
        "2026-07-02 12:45",
        "not recorded",
        "2026-07-02 13:15",
        "2026-07-02 13:30",
        "2026-07-02 13:45"
    ]
}

df_raw = pd.DataFrame(data)

df_raw

,order_id,customer_id,region,order_status,payment_amount,sales_rep,promo_code,order_time
0,ord-1001,C001,North,paid,1299.50,Li,SUMMER10,2026-07-02 09:00
1,ORD-1002,C002,north,PAID,850.00,li,,2026-07-02 09:15
2,ORD-1003,NaN,SOUTH,pending,NULL,Wang,none,2026-07-02 09:30
3,ORD-1004,N/A,south,NaN,399.90,wang,VIP20,2026-07-02 09:45
4,,C005,East,CANCELLED,-,Chen,,2026-07-02 10:00
5,ord-1006,C006,east,refunded,199.00,NULL,N/A,2026-07-02 10:15
6,ORD-1007,null,NaN,-,0,Zhao,WELCOME,2026-07-02 10:30
7,ORD-1008,C008,West,PAID,NaN,zhao,NaN,2026-02-30 14:20
8,ORD-1009,C009,west,pending,760.50,N/A,-,2026-07-02 11:00
9,NaN,NaN,central,unknown,500,Sun,NONE,2026-07-02 11:15


In [3]:
# 查看数据规模
df_raw.shape

(20, 8)

In [4]:
# 查看数据结构
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        19 non-null     str  
 1   customer_id     17 non-null     str  
 2   region          19 non-null     str  
 3   order_status    19 non-null     str  
 4   payment_amount  19 non-null     str  
 5   sales_rep       19 non-null     str  
 6   promo_code      18 non-null     str  
 7   order_time      20 non-null     str  
dtypes: str(8)
memory usage: 2.4 KB


In [5]:
# 建立字段列表
cols = [
    'order_id',
    'customer_id',
    'region',
    'order_status',
    'payment_amount',
    'sales_rep',
    'promo_code',
    'order_time'
]

In [6]:
# 查看各字段内容表达形式和数量
for col in cols:
    print(f"\n===={col}=====")
    print(df_raw[col].map(repr).value_counts(dropna=False))


====order_id=====
order_id
'ord-1001'      1
'ORD-1002'      1
' ORD-1003 '    1
'ORD-1004'      1
' '             1
'ord-1006'      1
'ORD-1007'      1
'ORD-1008'      1
'ORD-1009'      1
nan             1
'ORD-1011'      1
'ORD-1012'      1
'ord-1013'      1
'ORD-1014'      1
' ORD-1015 '    1
'ORD-1016'      1
'ORD-1017'      1
'ORD-1018'      1
'ORD-1019'      1
'ORD-1020'      1
Name: count, dtype: int64

====customer_id=====
customer_id
nan         3
'C001'      1
' C002 '    1
'N/A'       1
'C005'      1
'C006'      1
'null'      1
'C008'      1
'C009'      1
'C011'      1
''          1
'C013'      1
'C014'      1
'C015'      1
' none '    1
'C017'      1
'C018'      1
'C020'      1
Name: count, dtype: int64

====region=====
region
'SOUTH'        3
'North'        1
' north '      1
'south '       1
'East'         1
' east '       1
nan            1
'West'         1
' west '       1
'central'      1
' Central '    1
'N/A'          1
'NORTH'        1
'-'            1
'EAST'      

In [7]:
# 备份数据

df_cleaning = df_raw.copy()


In [8]:
# 建立假空值转化字典

missing_markers = {
    '':pd.NA,
    'N/A': pd.NA,
    'NULL':pd.NA,
    'none':pd.NA,
    'NONE':pd.NA,
    '-':pd.NA,
    'unknown' :pd.NA,
    'UNKNOWN':pd.NA
}

### 一、清洗 `order_id`

In [9]:
# 1.转为字符串类型、去除前后空格、统一大小写、规范化空值

df_cleaning['order_id'] = (
    df_cleaning['order_id']
    .astype('string')
    .str.strip()
    .str.upper()
    .replace(missing_markers)
)
df_cleaning['order_id']

0     ORD-1001
1     ORD-1002
2     ORD-1003
3     ORD-1004
4         <NA>
5     ORD-1006
6     ORD-1007
7     ORD-1008
8     ORD-1009
9         <NA>
10    ORD-1011
11    ORD-1012
12    ORD-1013
13    ORD-1014
14    ORD-1015
15    ORD-1016
16    ORD-1017
17    ORD-1018
18    ORD-1019
19    ORD-1020
Name: order_id, dtype: string

In [10]:
# 2.检查标准化之后的 order_id 是否有非空且不为规范格式的数据

invalid_order_id = (
    df_cleaning['order_id'].notna()
    & ~df_cleaning['order_id'].str.fullmatch(r"ORD-\d{4}")
)
invalid_order_id.sum()

np.int64(0)

In [11]:
# 统计order_id 的缺失值数量

order_id_missing_count = df_cleaning['order_id'].isna().sum()
order_id_missing_count

np.int64(2)

In [12]:
# 保存删除的行数据

removed_missing_order_id  = df_cleaning.loc[
    df_cleaning['order_id'].isna()
].copy()

removed_missing_order_id["remove_reason"] = "missing_order_id"

removed_missing_order_id

,order_id,customer_id,region,order_status,payment_amount,sales_rep,promo_code,order_time,remove_reason
4,<NA>,C005,East,CANCELLED,-,Chen,,2026-07-02 10:00,missing_order_id
9,<NA>,NaN,central,unknown,500,Sun,NONE,2026-07-02 11:15,missing_order_id


In [13]:
# 删除 order_id 缺失的行数据

df_cleaning = (
    df_cleaning
    .dropna(subset=["order_id"])
    .reset_index(drop=True)
)


In [14]:
# 验证当前数据是否还存在缺失值以及和删除数据的长度和是否等于原数据

assert(
    df_cleaning['order_id'].notna().all()
),"order_id中还存在缺失值"

assert(
    len(df_cleaning) + len(removed_missing_order_id) == len(df_raw)
),"当前保留记录与删除记录的总和无法对应原数据"

### 二、清洗 `customer_id`

In [15]:
# 1.转为字符串类型、去掉前后空格、统一大小写、规范化空值
df_cleaning['customer_id'] = (
    df_cleaning['customer_id']
    .astype('string')
    .str.strip()
    .str.upper()
    .replace(missing_markers)
)
df_cleaning['customer_id']

0     C001
1     C002
2     <NA>
3     <NA>
4     C006
5     <NA>
6     C008
7     C009
8     C011
9     <NA>
10    C013
11    C014
12    C015
13    <NA>
14    C017
15    C018
16    <NA>
17    C020
Name: customer_id, dtype: string

In [ ]:
# 2.检查标准化之后的 customer_id 是否还存在非空且不符合标准格式的数据

invalid_customer_id_mask = (
    df_cleaning['customer_id'].notna()
    & ~df_cleaning['customer_id'].str.fullmatch(r"\C\d{3}")
)

assert not invalid_customer_id_mask.any(), \
    "customer_id 中存在非空但格式不规范的数据"
invalid_customer_id_mask.sum()

np.int64(0)

In [ ]:
# 3.统计 customer_id 的空值数量

missing_customer_id_count = (
    df_cleaning['customer_id']
    .isna()
    .sum()
)
missing_customer_id_count

np.int64(6)